In [1]:
# Imports
import time

# Dependencies
import pandas as pd

# Library
from api_builder.fsm import menus
from api_builder.fsm.states import State
from api_builder.fsm.constants import NATIONS
from api_builder.fsm import fsm
from utils.utils import construct_rules
from api_builder.fsm.url_parser import parse_links
from api_builder.fsm.fetch import fetch_routine
from pipeline.builder import VanguardPipeline
from api_builder.fsm.query_builder import make_query
from data.vanguard_data import VanguardStorage
from api_builder.fsm.scrap import main_scrap_routine
from parsers.vanguard_parser import VanguardParser
from classifier.vanguard_classifier import VanguardClassifier
from api_builder.vanguard_api_build import MediaWikiAPI, VanguardScrapper
from api_builder.api_request import header
from cards.fsm						import CardFSM


In [2]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardParser(),
	VanguardStorage(),
	VanguardScrapper(web),
	VanguardClassifier()
)
await pipeline.scrapper.api.init_session()
state_machine = fsm.FSMContext()
state = State.ENTRY_POINT
while (state != State.END):
	if (state == State.ENTRY_POINT):
		state = menus.entry_point(state_machine)
	elif (state == State.SELECT_MAIN_CATEGORY):
		state = menus.select_category(state_machine)
	elif (state == State.SELECT_SUBCATEGORY):
		state = menus.select_subcategory(state_machine)
	elif (state == State.BUILD_QUERY):
		state = make_query(state_machine)
	elif (state == State.FETCH):
		state = await fetch_routine(state_machine, pipeline)
	elif (state == State.PARSE):
		pipeline.classifier._define_rules(construct_rules(state_machine.main_category.capitalize()))
		state = parse_links(state_machine, pipeline)
		state = State.END
time.sleep(4)

Welcome to VanguardTCGScrapper


What info do you need from the website?
0 : boosters
1 : specials
2 : decks
3 : others
4 : cards
You selected: boosters
Which subcategory you want to scrap?
indentify it with the index number:

0:  Booster Sets
1:  Extra Booster Sets
2:  Character Booster Sets
3:  Clan Booster Sets
4:  Title Booster Sets
5:  Unique Booster Sets


In [ ]:
ls = [[1], [2]]
len(ls)

In [ ]:
state_machine.data

In [ ]:
from data.check_data_base import build_set_path

for block in ["LB", "LL", "G", "V", "D", "DZ"]:
	consult = pipeline.parser.make_consults(getattr(pipeline.storage, block.lower()))
	print(consult)
	for tpl in consult.values():
		print(f"voy a pasar esta consulta: {tpl}")
		time.sleep(6)
		api_result = await pipeline.scrapper.api.get(
			params=tpl,
			headers=header
		)
		print(api_result)
		wikitex = pipeline.scrapper.obtain_wikitex(api_result)
		data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
		infobox = pipeline.parser.infobox(wikitex)
		is_d = False
		is_deck = False
		if (block in ["D", "DZ"]):
			is_d = True
		rows = pipeline.storage.prepare_data(data, 6, is_d=is_d, is_deck=is_deck, infobox=infobox)
		df = pd.DataFrame(rows, columns=[
			"CardNo", "Name", "Grade", "Faction",
			"FactionType", "Type", "Rarity", "Release"
		])
		set_number = pipeline.classifier.obtain_set_number(
			data[0]
		)
		path = build_set_path(
			category="boosters",
			set_type="main",
			block=block,
			set_number=set_number
		)
		path.parent.mkdir(
			parents=True,
			exist_ok=True
		)
		df.to_parquet(path)

		print(data)

In [17]:
db = pd.read_parquet("database/boosters/booster sets/v/set_08.parquet")
db.loc[db["Code"] == "None"]

,Code,Name,Grade,Faction,FactionType,Type,Rarity,Release,URL


In [18]:
db

,Code,Name,Grade,Faction,FactionType,Type,Rarity,Release,URL
0,V-BT08/001,Alter Ego Messiah,3,[Link Joker],Clan,Force,VR+SP,"June 19, 2020 (JP)September 4, 2020 (EN)",Alter Ego Messiah (V Series)
1,V-BT08/002,"Dragonic Overlord ""The X""",3,[Kagero],Clan,Force,VR+SP,"June 19, 2020 (JP)September 4, 2020 (EN)","Dragonic Overlord ""The X"" (V Series)"
2,V-BT08/003,"Dragonic Blademaster ""Souen""",3,[Kagero],Clan,Force,VR+SP,"June 19, 2020 (JP)September 4, 2020 (EN)","Dragonic Blademaster ""Souen"""
3,V-BT08/004,"Supreme Heavenly Battle Deity, Susanoo",3,[Oracle Think Tank],Clan,Protect,VR+SP,"June 19, 2020 (JP)September 4, 2020 (EN)","Supreme Heavenly Battle Deity, Susanoo (V Series)"
4,V-BT08/005,"Great Cosmic Hero, Grandgallop",3,[Dimension Police],Clan,Force,VR+SP,"June 19, 2020 (JP)September 4, 2020 (EN)","Great Cosmic Hero, Grandgallop (V Series)"
...,...,...,...,...,...,...,...,...,...
97,V-BT08/SP28,"Weather Forecaster, Miss Mist",0,[Oracle Think Tank],Clan,DrawPG,SP,"June 19, 2020 (JP)September 4, 2020 (EN)","Weather Forecaster, Miss Mist (V Series)"
98,V-BT08/SP30,Quick Shield,0,[],Clan,Blitz Order,SP,"June 19, 2020 (JP)September 4, 2020 (EN)",Quick Shield
99,V-BT08/SP31,Diamond Ace,0,[Dimension Police],Clan,DrawPG,SP,"June 19, 2020 (JP)September 4, 2020 (EN)",Diamond Ace (V Series)
100,V-BT08/SP33,Quick Shield,0,[],Clan,Blitz Order,SP,"June 19, 2020 (JP)September 4, 2020 (EN)",Quick Shield


In [ ]:
db[90:100]

In [ ]:
db[119:210]

In [3]:
pipeline.storage.v

['V Booster Set 01: Unite! Team Q4',
 'V Booster Set 02: Strongest! Team AL4',
 'V Booster Set 03: Miyaji Academy Cardfight Club',
 'V Booster Set 04: Vilest! Deletor',
 'V Booster Set 05: Aerial Steed Liberation',
 'V Booster Set 06: Phantasmal Steed Restoration',
 'V Booster Set 07: Infinideity Cradle',
 'V Booster Set 08: Silverdust Blaze',
 "V Booster Set 09: Butterfly d'Moonlight",
 'V Booster Set 10: Phantom Dragon Aeon',
 'V Booster Set 11: Storm of the Blue Cavalry',
 'V Booster Set 12: Divine Lightning Radiance']

In [4]:
dz_consults = pipeline.parser.make_consults(pipeline.storage.v, "consult")

In [6]:
dz_consults[8]

{'action': 'query',
 'format': 'json',
 'prop': 'revisions',
 'titles': "V Booster Set 09: Butterfly d'Moonlight",
 'rvprop': 'content',
 'rvslots': 'main'}

In [ ]:
dz_consults[0]

In [7]:
api_answer = await pipeline.scrapper.api.get(dz_consults[8], headers=header)

In [ ]:
api_answer["parse"]["text"]["*"]

In [8]:
api_answer

{'batchcomplete': '',
 'query': {'pages': {'1977459': {'pageid': 1977459,
    'ns': 0,
    'title': "V Booster Set 09: Butterfly d'Moonlight",
    'revisions': [{'slots': {'main': {'contentmodel': 'wikitext',
        'contentformat': 'text/x-wiki',
        '*': '{{Infobox\n|Box title = VGE-V-BT09:Butterfly d\'Moonlight\n|image = File:VGE-V-BT09.png\n|Row 1 title = Japanese Name\n|Row 1 info = {{Ruby|蝶魔月影|ちょうまげつえい}}\n|Row 2 title = Phonetic\n|Row 2 info = Chōma Getsuei\n|Row 3 title = Translation\n|Row 3 info = Butterfly & Magic Under Moon\'s Shadow\n|Row 4 title = Release Date\n|Row 4 info = July 31, 2020 (JP)<br>October 2, 2020 (EN)\n|Row 5 title = Set Gallery Japanese\n|Row 5 info = [[Set Gallery:VG-V-BT09|VG-V-BT09]]\n|Row 6 title = Set Gallery English\n|Row 6 info = [[Set Gallery:VGE-V-BT09|VGE-V-BT09]]\n|Row 7 title = Previous Set\n|Row 7 info = [[V Booster Set 08: Silverdust Blaze|Silverdust Blaze]]\n|Row 8 title = Next Set\n|Row 8 info = [[V Booster Set 10: Phantom Dragon Aeon|P

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
soup = BeautifulSoup(api_answer["parse"]["text"]["*"], "html.parser")

In [9]:
wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
infobox = pipeline.parser.infobox(wikitex)

In [13]:
card_fsm = CardFSM(state_machine)

In [14]:
data[95]

'{{CardListV|V-BT09/SP19|Hades Hypnotist|V|0|Pale Moon|DrawPG|SP}}'

In [ ]:
import unicodedata
def clean_text(text: str) -> str:
	text = unicodedata.normalize("NFKC", text)

	invisible_chars = [
		"\u200e",
		"\u200f",
		"\u200b",
		"\ufeff"
	]

	for char in invisible_chars:
		text = text.replace(char, "")

	return (text.strip())

In [ ]:
clean_text(str(data[15].params[1].value).strip())

In [ ]:
data[0].params[1]

In [ ]:
wiki = pipeline.scrapper.obtain_links(api_answer)

In [ ]:
pipeline.parser.clean_trash_from_set(state_machine.data["page"], wiki, 4, reverse=True)

In [ ]:
data[0].params[1].value

In [ ]:
pipeline.storage.lb

In [ ]:
type(wiki[0])

In [ ]:
param = {
    "action": "parse",
    "page": "V Booster Set 01: Unite! Team Q4",
    "prop": "links",
    "format": "json"
}

In [ ]:
api_answer = await pipeline.scrapper.api.get(params=param, headers=header)

In [ ]:
api_answer

In [ ]:
print(pipeline.storage.prepare_data([data[0]], card_fsm))

In [ ]:
df = pd.DataFrame(data, columns=[
	"CardNo", "Name", "Grade", "Faction",
	"FactionType", "Type", "Rarity", "Release"
])

In [ ]:
df

In [ ]:
await main_scrap_routine(pipeline.parser, pipeline.storage, pipeline.scrapper, pipeline.classifier)

In [ ]:
api_answer["query"]["pages"]["525299"]["revisions"][0]["slots"]["main"]["*"]

In [ ]:
api_result = await pipeline.scrapper.api.get(
	params=dz_consults[0],
	headers=header
)

In [ ]:
api_result

In [ ]:
df = pd.read_parquet("data/database/boosters/booster sets/v/set_-1_8.parquet")

In [ ]:
df